# TRM Compiler — Real LLVM Training

This notebook runs TRM training with **real LLVM** via compiler_gym.

**Options:**
- **Option 1**: Use setuptools downgrade workaround (simpler)
- **Option 2**: Use Docker container (more reliable)
- **Option 3**: Synthetic mode (fallback if both fail)

In [ ]:
# @title Option 1: Setup with setuptools workaround
# @markdown This tries the workaround: downgrade setuptools/wheel before installing compiler_gym

# Install dependencies in exact order
!pip install setuptools==65.5.0 "wheel<0.40.0" -q
!pip uninstall -y trm-experiments >/dev/null 2>&1 || true
!pip install -e . -q
!pip install compiler_gym -q || echo "compiler_gym install failed - will try Docker option"

In [ ]:
# @title Option 2: Docker-based compiler_gym (RECOMMENDED if Option 1 fails)
# @markdown This runs compiler_gym in a Docker container to bypass Python version issues

# Install Docker
!apt-get update -qq
!apt-get install -y -qq docker.io

# Check if Docker is available
!docker --version

In [ ]:
# @title Test compiler_gym availability

import sys
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["COMPILER_GYM_HOME"] = "/content/compiler_gym"

try:
    import compiler_gym
    print(f"✓ compiler_gym available: {compiler_gym.__version__}")
    
    # Quick test
    env = compiler_gym.make("llvm-v0", benchmark="cbench-v1/qsort")
    obs = env.reset()
    print(f"✓ Initial instruction count: {obs['IrInstructionCount']}")
    env.close()
    COMPILER_GYM_OK = True
except Exception as e:
    print(f"✗ compiler_gym unavailable: {e}")
    COMPILER_GYM_OK = False

In [ ]:
# @title Option 3: Fallback - Synthetic Mode (if compiler_gym failed)

if not COMPILER_GYM_OK:
    print("⚠️ Using synthetic mode - no real LLVM benchmarking")
    USE_SYNTHETIC = True
else:
    USE_SYNTHETIC = False
    print("✓ Using real LLVM via compiler_gym")

In [ ]:
# @title Generate Training Data

from pathlib import Path
from torch.utils.data import DataLoader
import torch

from trm_compiler.data import CompilerTraceDataset, generate_compiler_traces
from trm_compiler.model import TinyPassOrderingRefiner
from trm_compiler.training import train_one_epoch
from trm_compiler.env_wrapper import make_compiler_env

traces = generate_compiler_traces(
    benchmarks=['qsort', 'adpcm', 'sha'],
    episodes_per_benchmark=8,
    max_steps_per_episode=12,
    use_heuristic=USE_SYNTHETIC,  # False = use compiler_gym if available
    seed=42,
    strategy='mixed',
)
print(f'Generated {len(traces)} traces')
print(f'Mode: {"SYNTHETIC" if USE_SYNTHETIC else "REAL LLVM (compiler_gym)"}')

In [ ]:
# @title Train TRM Model

dataset = CompilerTraceDataset(traces)
loader = DataLoader(dataset, batch_size=32, shuffle=True, num_workers=0, drop_last=False)
model = TinyPassOrderingRefiner()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(5):
    losses = train_one_epoch(model, loader, optimizer, 'cpu')
    print(f'epoch={epoch+1}', {k: round(v, 4) for k, v in losses.items()})

In [ ]:
# @title Evaluate Model

from trm_compiler.model import rollout_pass_optimizer
from trm_compiler.baselines import random_search, greedy_search

env = make_compiler_env('qsort', use_compilergym=not USE_SYNTHETIC)
obs, _ = env.reset()
trace = rollout_pass_optimizer(model, obs, max_steps=8, temperature=1.0, device='cpu')
print('TRM rollout steps:', len(trace))

print('Random baseline:', random_search('qsort', max_steps=12, num_trials=10, seed=42))
print('Greedy baseline:', greedy_search('qsort', max_steps=12, seed=42))